# 🧩 Functions — advanced concepts

**Time:** ~30 minutes &nbsp;•&nbsp; **Goal:** turn a pile of small functions into a real **analysis** — and use it to watch evolution happen.

You can already `translate()` a gene. Now you'll **document** your tools the way professionals do, have functions **return several answers at once**, make them **call each other**, and finish by measuring **genetic drift**: which mutations change the protein, and which are silent?

Skills:

1. Docstrings & type hints — say what a function does
2. Return several values — a whole result in one call
3. Functions calling functions — compose an analysis
4. **Finale:** synonymous vs missense — drift, made visible 🏆

## Our data — one gene, two species

Same setup as the basics notebook. Run it first.

In [ ]:
# The genetic code: every three-letter codon -> its amino acid (one letter).
codon_table = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}
stop_codons = {"TAA", "TAG", "TGA"}

# The same short gene, sampled from two related species.
ancestor  = "ATGAAAGGTTTTCTGGAATATGCAGGCTAA"   # species A - the demos use this
species_b = "ATGAAAGGCTTCCTGGAGTATGCAAGCTAA"   # species B - your DIY challenges use this

print("species A:", ancestor)
print("species B:", species_b)


# our translate() from the basics notebook - we'll build on it here
def translate(seq, frame=0):
    protein = ""
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        protein = protein + codon_table[codon]
    return protein

## Skill 1 — Docstrings & type hints

Two habits make a function readable months later:

- a **type hint** says what goes in and what comes out: `def gc_content(seq: str) -> float:`
- a **docstring** — a `"""triple-quoted"""` line right under `def` — says *what it does*

```python
def gc_content(seq: str) -> float:
    """Return the percentage of bases that are G or C."""
    ...
```

The code runs exactly the same; the difference is that a human (and your editor) now knows what it's for. `str -> float`: takes text, gives a number.

In [ ]:
# DEMO - the same gc_content, now documented
def gc_content(seq: str) -> float:
    """Return the percentage of bases in seq that are G or C, to 1 dp."""
    gc = 0
    for base in seq:
        if base in "GC":
            gc = gc + 1
    return round(100 * gc / len(seq), 1)

help(gc_content)                    # the docstring shows up here!
print("species A GC%:", gc_content(ancestor))

### 🧩 DIY challenge

Add documentation to your own `translate`:

1. Write a version of `translate` with a type hint: `def translate(seq: str, frame: int = 0) -> str:`.
2. Give it a one-line docstring saying what it does.
3. Run `help(translate)` to see your docstring, then translate `species_b` to prove it still works.

In [ ]:
# TODO - your turn

## Skill 2 — Return several values at once

A function can hand back a whole **package** of results. The tidiest way is to `return` a **dictionary** — labelled answers you can look up by name:

```python
return {"length": ..., "gc": ..., "protein": ...}
```

One call, a complete summary. (You met dictionaries in Lab 2 — key → value.)

In [ ]:
# DEMO - one call, a full summary of a gene
def analyse(seq: str) -> dict:
    """Return length, GC%, and protein for a DNA sequence."""
    return {
        "length": len(seq),
        "gc": gc_content(seq),
        "protein": translate(seq),
    }

result = analyse(ancestor)
print(result)
print("just the protein:", result["protein"])

### 🧩 DIY challenge

Using `analyse` on **species B** (`species_b`):

1. Call `analyse(species_b)` and store it in a variable.
2. Print its `protein` and its `gc` on separate lines, looking each up by key.
3. Compare its `gc` to species A's — has the gene drifted GC-richer or GC-poorer?

In [ ]:
# TODO - your turn

## Skill 3 — Functions calling functions

Real programs are **small functions stacked together**. A bigger function just *calls* the ones you already trust. Here `count_differences` compares two proteins by calling `translate` twice:

```python
def count_differences(seq_a, seq_b):
    prot_a = translate(seq_a)
    prot_b = translate(seq_b)
    ...
```

You don't re-explain how translation works — you *reuse* it. That's the whole idea.

In [ ]:
# DEMO - how many amino acids differ between two genes?
def count_differences(seq_a: str, seq_b: str) -> int:
    """Count positions where the two translated proteins disagree."""
    prot_a = translate(seq_a)
    prot_b = translate(seq_b)
    differences = 0
    for a, b in zip(prot_a, prot_b):        # walk both proteins together
        if a != b:
            differences = differences + 1
    return differences

print("protein differences A vs B:", count_differences(ancestor, species_b))

### 🧩 DIY challenge

Write a sister function `count_dna_differences(seq_a, seq_b)`:

1. It should compare the **DNA** directly (no translation) — `zip(seq_a, seq_b)`, count mismatches.
2. Call it on `ancestor` and `species_b`.
3. Compare that number with `count_differences(ancestor, species_b)`. There are **more DNA changes than protein changes** — hold that thought for the finale.

In [ ]:
# TODO - your turn

## Skill 4 — Finale: synonymous vs missense 🏆

Here's the payoff. Most DNA mutations are **synonymous** ("silent") — they change a codon but *not* the amino acid, because the genetic code is redundant. The rest are **missense** — they change the protein, and natural selection can *see* those.

You'll write one function that changes a single base and reports **which kind of mutation** it is — the atom of molecular evolution:

```python
def classify_mutation(seq, position, new_base):
    protein_before = translate(seq)
    # ...swap in the new base...
    protein_after = translate(mutated)
    return "synonymous" if protein_after == protein_before else "missense"
```

In [ ]:
# DEMO - is a point mutation silent, or does it change the protein?
def classify_mutation(seq: str, position: int, new_base: str) -> str:
    """Swap one base and report whether the protein changes."""
    protein_before = translate(seq)
    mutated = seq[:position] + new_base + seq[position + 1:]   # rebuild with one base changed
    protein_after = translate(mutated)
    if protein_after == protein_before:
        return "synonymous (silent)"
    return "missense (protein changed)"

# position 8 is the 3rd base of codon 3 (GGT -> GGG, still Glycine)
print("pos 8  T->G:", classify_mutation(ancestor, 8, "G"))
# position 6 is the 1st base of codon 3 (GGT -> AGT, Gly -> Ser)
print("pos 6  G->A:", classify_mutation(ancestor, 6, "A"))

### 🧩 DIY challenge

Explore drift in **species B** (`species_b`) with your own `classify_mutation`:

1. Pick a position in a **third** codon position (like `8`, `11`, `14` …) and mutate it — these are *often* synonymous. Confirm the label.
2. Pick a **first** codon position (like `6`, `9`, `12` …) and mutate it — these are *usually* missense. Confirm.
3. In one sentence: why does the *position within a codon* predict whether a mutation is silent?

In [ ]:
# TODO - your turn

## 🔭 Explore further

*(Helfrid will drop free web resources here.)*

- `*args` and `**kwargs` — functions that take *any* number of arguments. **[link]**
- `lambda` — tiny throwaway functions, e.g. `sorted(genes, key=lambda g: gc_content(g))`. **[link]**
- **Pure** functions (same input → same output, no side effects) vs functions that change things — why the pure ones are easier to test. **[link]**

## ✅ You've got it if you can…

- [ ] Add a type hint and a docstring to a function, and read them back with `help()`
- [ ] Return a dictionary of several results from one function
- [ ] Write a function that works by **calling** other functions
- [ ] Explain the difference between a **synonymous** and a **missense** mutation
- [ ] Use `classify_mutation` to test a point mutation yourself

**Next:** real sequence files are messy — missing bases, wrong lengths, odd letters. Open **`03_errors_basics.ipynb`** to make your tools survive them.